# MLflow Intro Notebook
In this example we explore utilizing MLflow to track our ML experiments. Specifically we take the famous [Iris classification dataset](https://www.kaggle.com/datasets/arshid/iris-flower-dataset) and train three different classifier models on the data.

We showcase how you create an experiment and create nested runs within that experiment to track all models. As follow-up in future NBs you can also understand how to register this model in Unity Catalog and deploy to a Databricks Serving Endpoint.

### Additional Resources/Credit
- MLflow Intro: https://mlflow.org/docs/latest/ml/getting-started/
- Databricks Samples: https://github.com/RamVegiraju/databricks-samples/tree/master

## Setup

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

## Dataset & Model Preparation

In [0]:
# 1) Dataset
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2) Define models to compare (same data, different algorithms)
candidates = {
    "logreg": LogisticRegression(max_iter=200, solver="lbfgs", multi_class="auto"),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "grad_boost": GradientBoostingClassifier(random_state=42),
}

## MLflow Experiment Setup

In [0]:
# first is specific to being in a DB notebook/workspace
# if not in workspace can set simpler as: mlflow.set_experiment("iris-model-comparison")
user = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
)
mlflow.set_experiment(f"/Users/{user}/iris-model-comparison")

In [0]:
results = []
best = {"run_id": None, "metric": -1.0, "name": None}

# 3) Track a parent run + one child run per model (easy to compare in UI)
with mlflow.start_run(run_name="iris_compare_parent") as parent:
    mlflow.log_param("dataset", "iris")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    for name, model in candidates.items():
        with mlflow.start_run(run_name=name, nested=True) as run:
            # Log model hyperparameters
            mlflow.log_param("model_name", name)
            mlflow.log_params(model.get_params())

            # Train + evaluate
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds, average="macro")

            # Log metrics
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_macro", f1)

            # Log model artifact
            signature = infer_signature(X_train, model.predict(X_train))
            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="model",
                signature=signature,
                input_example=X_train[:5],
                registered_model_name=None,  # optionally register later
            )

            results.append((name, run.info.run_id, acc, f1))

            # Track best by accuracy (change to f1_macro if you prefer)
            if acc > best["metric"]:
                best = {"run_id": run.info.run_id, "metric": acc, "name": name}
                print(best)